# Bag-of-Words SL (word-heteroskedastic)

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
    mse_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sl.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsSLConfig.get_canonical(
    dataset="word_heteroskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-word-het-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
)

display(config.visualize())

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,7737.097656,928.161072,49933.394531,49984.0,7737.097656,986.280212,2003.641113,49984.0,2265.868408,1415.594727,50076.695312,49920.0,2265.868408,1333.998291,2002.146606,49920.0
1,4482.124023,1742.899414,49946.277344,49984.0,4482.124023,1598.797363,2003.92395,49984.0,7243.936523,2454.241211,50076.695312,49920.0,7243.936523,2337.871826,2002.146606,49920.0


In [5]:
analysis = BagOfWordsAnalysisConfig.from_grouped({
    "example": [(0, config.study_folder)]
})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])

In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.722656,-0.367188,-0.098633
-0.19043,0.0859375,-0.933594
-0.253906,0.03125,0.691406
-0.214844,0.0,0.096191
-0.298828,0.015625,-0.429688
